In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to output dir
output_path = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary.csv"

# Path to zarr
local_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import with_tissue_artifact

# Path to cache file
cache_file = "cache_all_slides.pkl"

df_tissue = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="complete", version="default")
with_tissue = list(set(df_tissue["filename"].tolist()))
print("WSIs with completed default tissue detection: ", len(with_tissue))

df_sub = df_all[df_all["filename"].isin(with_tissue)].copy()

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_sub, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = subset_df_list(df_HE, "T_category", "Blood, Bone Marrow and Lymphatic System")
df_HE = df_HE[df_HE['M_category'].apply(len) == 1]

In [ ]:
all_filenames = df_HE["filename"].tolist()

print("Number of files: ", len(all_filenames))

In [ ]:
from feature_extraction import ExtractMany

# NOT CONTINUED: h-optimus-0, titan (due to processing time)
# DONE: conch, uni
# TRY: uni2
# TRY: without artifacts
extractor = ExtractMany(all_filenames, output_path, local_zarr_dir = local_dir, model = "conch", remove_artifacts = False)